# **1. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [61]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import randint

# **2. Memuat Dataset dari Hasil Clustering**

Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [62]:
df = pd.read_csv("Dataset_clustering.csv")

df.head()

,TransactionAmount,TransactionDuration,AccountBalance,CustomerAge,TransactionType,Channel,Cluster
0,14.09,81.0,5112.21,70.0,Debit,ATM,2
1,376.24,141.0,13758.91,68.0,Debit,ATM,2
2,126.29,56.0,1122.35,19.0,Debit,Online,3
3,184.50,25.0,8569.06,26.0,Debit,Online,3
4,13.45,198.0,7429.40,26.0,Credit,Online,3


# **3. Data Splitting**

Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [63]:
le_type = LabelEncoder()
df['TransactionType'] = le_type.fit_transform(df['TransactionType'])

le_channel = LabelEncoder()
df['Channel'] = le_channel.fit_transform(df['Channel'])

In [64]:
X = df.drop(columns=['Cluster'])
y = df['Cluster'] - df['Cluster'].min()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# **4. Membangun Model Klasifikasi**


## **a. Membangun Model Klasifikasi**

Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Pilih algoritma klasifikasi yang sesuai, seperti Logistic Regression, Decision Tree, Random Forest, atau K-Nearest Neighbors (KNN).
2. Latih model menggunakan data latih.

In [65]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


In [66]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:35:27] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Tulis narasi atau penjelasan algoritma yang Anda gunakan.

## **b. Evaluasi Model Klasifikasi**

Berikut adalah **rekomendasi** tahapannya.
1. Lakukan prediksi menggunakan data uji.
2. Hitung metrik evaluasi seperti Accuracy dan F1-Score (Opsional: Precision dan Recall).
3. Buat confusion matrix untuk melihat detail prediksi benar dan salah.

In [67]:
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Recall:", recall)

print("Random Forest Performance:\n", classification_report(y_test, y_pred_rf))

Accuracy: 1.0
F1 Score: 1.0
Recall: 1.0
Random Forest Performance:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       174
           1       1.00      1.00      1.00       167
           2       1.00      1.00      1.00       162

    accuracy                           1.00       503
   macro avg       1.00      1.00      1.00       503
weighted avg       1.00      1.00      1.00       503



In [68]:
y_pred = xgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Recall:", recall)

print("XGBoost Performance:\n", classification_report(y_test, y_pred_xgb))

Accuracy: 1.0
F1 Score: 1.0
Recall: 1.0
XGBoost Performance:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       174
           1       1.00      1.00      1.00       167
           2       1.00      1.00      1.00       162

    accuracy                           1.00       503
   macro avg       1.00      1.00      1.00       503
weighted avg       1.00      1.00      1.00       503



Tulis hasil evaluasi algoritma yang digunakan, jika Anda menggunakan 2 algoritma, maka bandingkan hasilnya.

## **c. Tuning Model Klasifikasi (Optional)**

Gunakan GridSearchCV, RandomizedSearchCV, atau metode lainnya untuk mencari kombinasi hyperparameter terbaik

In [69]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='f1_macro')
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

In [70]:
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0]
}
random_search = RandomizedSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
                                   param_distributions=param_dist,
                                   n_iter=20, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)
best_xgb = random_search.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:37:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


## **d. Evaluasi Model Klasifikasi setelah Tuning (Optional)**

Berikut adalah rekomendasi tahapannya.
1. Gunakan model dengan hyperparameter terbaik.
2. Hitung ulang metrik evaluasi untuk melihat apakah ada peningkatan performa.

In [71]:
print("Tuned Random Forest Performance:\n", classification_report(y_test, y_pred_best_rf))

accuracy = accuracy_score(y_test, y_pred_best_rf)
f1 = f1_score(y_test, y_pred_best_rf, average='weighted')
recall = recall_score(y_test, y_pred_best_rf, average='weighted')

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Recall:", recall)

Tuned Random Forest Performance:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       174
           1       1.00      1.00      1.00       167
           2       1.00      1.00      1.00       162

    accuracy                           1.00       503
   macro avg       1.00      1.00      1.00       503
weighted avg       1.00      1.00      1.00       503

Accuracy: 1.0
F1 Score: 1.0
Recall: 1.0


In [72]:
print("Tuned XGBoost Performance (RandomizedSearchCV):\n", classification_report(y_test, y_pred_best_xgb))

accuracy = accuracy_score(y_test, y_pred_best_xgb)
f1 = f1_score(y_test, y_pred_best_xgb, average='weighted')
recall = recall_score(y_test, y_pred_best_xgb, average='weighted')

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Recall:", recall)

Tuned XGBoost Performance (RandomizedSearchCV):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       174
           1       1.00      1.00      1.00       167
           2       1.00      1.00      1.00       162

    accuracy                           1.00       503
   macro avg       1.00      1.00      1.00       503
weighted avg       1.00      1.00      1.00       503

Accuracy: 1.0
F1 Score: 1.0
Recall: 1.0


## **e. Analisis Hasil Evaluasi Model Klasifikasi**

Berikut adalah **rekomendasi** tahapannya.
1. Bandingkan hasil evaluasi sebelum dan setelah tuning (jika dilakukan).
2. Identifikasi kelemahan model, seperti:
  - Precision atau Recall rendah untuk kelas tertentu.
  - Apakah model mengalami overfitting atau underfitting?
3. Berikan rekomendasi tindakan lanjutan, seperti mengumpulkan data tambahan atau mencoba algoritma lain jika hasil belum memuaskan.

### 1. Perbandingan hasil evaluasi sebelum dan setelah tuning

Hasil evaluasi sebelum dan setelah tuning tidak mengalami perubahan. Model tetap memiliki skor sempurna (1.0) di semua metrik.

### 2. Identifikasi kelemahan model

Model kemungkinan besar overfitting terhadap data latih. Kedua model memiliki akurasi 100% di training dan validation set. Hal ini mengindikasikan bahwa model telah mengingat pola data secara sempurna, sehingga sulit untuk diuji pada data baru yang berbeda. Karena model mendapatkan skor sempurna, sulit untuk mengetahui apakah model benar-benar bisa mengenali pola baru atau hanya mengingat data yang ada.

### 3. Rekomendasi tindakan lanjutan

- Menggunakan lebih banyak data dan data yang lebih beragam untuk membangun model klasifikasi
-  Periksa apakah dataset memiliki class imbalance dan atasi dengan teknik yang sesuai. Contohnya dengan teknik SMOTE
- Jika dua fitur sangat mirip, hapus salah satunya untuk menghindari informasi berlebih yang menyebabkan overfitting.
-  Lakukan feature selection agar model tidak terlalu bergantung pada fitur tertentu.